# CURATOR - Phase B step-time and vLLM probe

Same setup as the pilot notebook: **GPU T4 x2**, **Internet On**, dataset `curator-rl-bundle` attached
(rebuild the zip first: `python scripts/make_kaggle_bundle.py`, then upload a new dataset version).

Answers: how long one GRPO optimizer step takes at the real batch shape (8 and 16 prompts x 8 samples,
512 tokens), how that splits into generation and update, and whether vLLM works on a T4.
Run cells 1-3 first (about 10-15 minutes). Cell 4 (vLLM) is optional and may take longer.

In [ ]:
# 1) Setup: unpack the bundle, pin TRL
import os, shutil, glob, zipfile
WORK = "/kaggle/working/curator-rl"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
zips = glob.glob("/kaggle/input/**/curator-rl-bundle.zip", recursive=True)
if zips:
    os.makedirs(WORK); zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    assert roots, "attach the curator-rl-bundle dataset"
    shutil.copytree(roots[0], WORK)
assert not os.path.exists(WORK + "/data/test_sealed")
!pip uninstall -y -q torchao
!pip install -q pydantic pyyaml "trl==1.15.0"
%cd /kaggle/working/curator-rl
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 2) The measurement: generation floor + real GRPO steps (HF backend), GPU 0
!CUDA_VISIBLE_DEVICES=0 python scripts/kaggle_step_probe.py --out /kaggle/working/step_probe --prompts-per-step 8,16 --steps 3

In [ ]:
# 3) Print the summary again (copy this back to the chat)
import json
d = json.load(open("/kaggle/working/step_probe/step_probe.json"))
for name, s in d["stages"].items():
    print(name, "ok" if s["ok"] else "FAILED " + s["error"])
    if s["ok"]:
        print(json.dumps(s["result"], indent=1)[:3500])

## 4) Optional: vLLM on T4
This installs vLLM into a separate virtual environment (it pins its own torch, so it must not touch the
kernel's). It downloads several GB and can take 10+ minutes. If it fails, the error text is the answer.

In [ ]:
%%bash
cd /kaggle/working/curator-rl
pip install -q virtualenv && python -m virtualenv -q /kaggle/working/venv_vllm
/kaggle/working/venv_vllm/bin/pip install -q vllm "trl==1.15.0" peft datasets pydantic pyyaml numpy scipy pandas pyarrow 2>&1 | tail -n 5
CUDA_VISIBLE_DEVICES=1 /kaggle/working/venv_vllm/bin/python scripts/kaggle_step_probe.py \
    --backend vllm --out /kaggle/working/step_probe_vllm --prompts-per-step 16 --steps 3 2>&1 | tail -n 25